# 05 — Local/Colab Baselines (Task 2)

Run `NaiveTissueDrugMeanPredictor`, `RandomForest`, and `MultiViewXGBoost` from
drevalpy's `MODEL_FACTORY` under all four splits — no reimplementation. Runs
locally (CPU) or on Colab — detected automatically below.

**Important findings (verified against the installed drevalpy 1.5.1 source, not assumed):**
- `drug_response_experiment()` generates and manages its own train/validation/test
  splits internally via `response_data.split_dataset(mode=test_mode, split_validation=True,
  validation_ratio=0.1, ...)`. This is **separate** from the `data/splits/*.json` files
  built in notebook 04 — those aren't consumed here.
- `test_mode` is a single string, not a list — needs four separate calls.
- **If saved splits already exist for a given `run_id` + `test_mode`, drevalpy reuses
  them instead of regenerating.** This notebook and `06_baselines_gpu.ipynb` share the
  same `RUN_ID` on purpose — whichever runs first creates the canonical splits for each
  test mode, and the other reuses them, guaranteeing every model (local and GPU) is
  evaluated on identical folds. Run this one (05) first to keep it deterministic.
- `xgboost` is an optional drevalpy extra, not a base dependency — installed explicitly
  below.

## Environment setup (Colab or local)

Detects Colab automatically. On Colab, mounts Drive and expects (or will create)
project data under `MyDrive/multiomics_project/` — same `BASE_PATH` convention as
`00_colab_setup.ipynb` from Session 2. If `data/GDSC2/GDSC2.csv` isn't there yet,
drevalpy will auto-download it from Zenodo on first use (slower, self-healing —
no need to manually upload first if you'd rather let it fetch fresh).

In [1]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q "drevalpy[xgboost]"
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Running on local | BASE_PATH = ..


## Imports and config

In [2]:
import time
from glob import glob

import pandas as pd
from scipy.stats import pearsonr

from drevalpy.datasets.loader import load_dataset
from drevalpy.experiment import drug_response_experiment
from drevalpy.models import MODEL_FACTORY

In [3]:
DATA_DIR = BASE_PATH / "data"          # resolves to <DATA_DIR>/GDSC2/GDSC2.csv internally
RESULTS_DIR = BASE_PATH / "results"
RUN_ID = "session4_baselines"          # shared with 06_baselines_gpu.ipynb — see note above

N_CV_SPLITS = 5
TEST_MODES = ["LPO", "LCO", "LDO", "LTO"]

# load_dataset() defaults to "LN_IC50_curvecurator" (a refit measure) — explicit
# here to match the raw LN_IC50 used throughout README/PLAN/notebooks 02-04.
RESPONSE_MEASURE = "LN_IC50"

# True = real hyperparameter search per model per fold (slower, the honest comparison).
# False = first hyperparameter set only (fast smoke test). Flip to True once this runs clean.
HYPERPARAMETER_TUNING = True

## Load the GDSC2 response dataset

In [4]:
response_data = load_dataset("GDSC2", path_data=str(DATA_DIR), measure=RESPONSE_MEASURE)
print(f"{response_data.dataset_name}: {len(response_data)} responses")

/home/shoko/miniconda3/envs/multiomics/lib/python3.11/site-packages/drevalpy/datasets/loader.py:62: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  response_data = pd.read_csv(path, dtype={"pubchem_id": str, "cell_line_name": str})


GDSC2: 234436 responses


## Select models from `MODEL_FACTORY`

`RandomForest` (gene-expression-only) is the primary baseline to beat.
`MultiViewXGBoost` is secondary context — different omics mix (CNV+methylation+mutation,
not protein). `NaiveTissueDrugMeanPredictor` is the naive baseline; `NaiveMeanEffectsPredictor`
gets added automatically by `drug_response_experiment` regardless of what's passed here.

In [5]:
models = [MODEL_FACTORY["RandomForest"], MODEL_FACTORY["MultiViewXGBoost"]]
baselines = [MODEL_FACTORY["NaiveTissueDrugMeanPredictor"]]

print("Models:   ", [m.__name__ for m in models])
print("Baselines:", [b.__name__ for b in baselines])

Models:    ['RandomForest', 'MultiViewXGBoost']
Baselines: ['NaiveTissueDrugMeanPredictor']


## Run the experiment — one call per split type

`overwrite=False`, so if this gets interrupted, rerunning skips folds/splits already
on disk instead of redoing them.

In [6]:
for test_mode in TEST_MODES:
    print(f"\n========== {test_mode} ==========")
    start = time.time()
    drug_response_experiment(
        models=models,
        baselines=baselines,
        response_data=response_data,
        test_mode=test_mode,
        n_cv_splits=N_CV_SPLITS,
        run_id=RUN_ID,
        path_out=str(RESULTS_DIR),
        path_data=str(DATA_DIR),
        hyperparameter_tuning=HYPERPARAMETER_TUNING,
        overwrite=False,
    )
    elapsed = time.time() - start
    print(f"{test_mode} done in {elapsed / 60:.1f} min")


========== LPO ==========
Creating cv splits at ../results/session4_baselines/GDSC2/LPO/splits
Running RandomForest
- Full Test -

################# FOLD 1/5 #################

Training model with hyperparameters: {'cell_line_views': 'gene_expression', 'criterion': 'squared_error', 'drug_views': 'fingerprints', 'max_depth': 5, 'max_samples': 0.2, 'n_estimators': 100, 'n_jobs': -1}
Loading cell line features ...
Loading a RandomForest with the following cell line views: ['gene_expression']
Loading drug features ...
Loading a RandomForest with the following drug views: ['fingerprints']
Number of cell lines in features: 1010
Number of drugs in features: 287
Number of cell lines in train dataset: 969
Number of drugs in train dataset: 285
Reduced training dataset from 167493 to 163763, due to missing features
Reduced prediction dataset from 18611 to 18191, due to missing features
Training model ...
Using temporary directory: /tmp/tmprig1swpt for model checkpoints
current best RMSE score: 2

KeyboardInterrupt: 

## Quick sanity check: per-fold Pearson r

Just a smoke test that runs produced sane, non-degenerate predictions — full
model x split x metric consolidation (with RMSE, per-tissue breakdown, and the
GPU models from Task 3) happens in Task 4.

In [ ]:
rows = []
for test_mode in TEST_MODES:
    mode_dir = RESULTS_DIR / RUN_ID / response_data.dataset_name / test_mode
    for pred_file in sorted(glob(str(mode_dir / "*" / "predictions" / "predictions_split_*.csv"))):
        model_name = Path(pred_file).parts[-3]
        fold = int(Path(pred_file).stem.rsplit("_", 1)[-1])
        df = pd.read_csv(pred_file).dropna(subset=["response", "predictions"])
        r, _ = pearsonr(df["predictions"], df["response"])
        rows.append({"split": test_mode, "model": model_name, "fold": fold, "n": len(df), "pearson_r": round(r, 4)})

summary = pd.DataFrame(rows)
summary.groupby(["split", "model"])["pearson_r"].agg(["mean", "std", "count"])